# Homework 3 — Classification

Predict `converted` (did the lead sign up for the platform?) with logistic regression.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

Check missing values, then fill: categorical → `'NA'`, numerical → `0.0`.

In [2]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [3]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
target = 'converted'

# pandas 3 stores strings as the 'str' dtype, so select numbers instead of 'object'
numerical = [c for c in df.select_dtypes('number').columns if c != target]
categorical = [c for c in df.columns if c not in numerical and c != target]

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

print('categorical:', categorical)
print('numerical:', numerical)
df.isnull().sum().sum()

categorical: ['lead_source', 'industry', 'employment_status', 'location']
numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


np.int64(0)

## Question 1 — mode of `industry`

In [5]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [6]:
df.industry.mode()[0]

'technology'

## Question 2 — correlation matrix of numerical features

In [7]:
corr = df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [8]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(f'{a} / {b}: {corr.loc[a, b]:.4f}')

interaction_count / lead_score: 0.9157
number_of_courses_viewed / lead_score: 0.7572
number_of_courses_viewed / interaction_count: 0.7216
annual_income / interaction_count: 0.1228


## Split the data

60/20/20 with the exact calls from the homework.

In [9]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Question 3 — mutual information (training set only)

In [10]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False).round(2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Question 4 — logistic regression with one-hot encoding

In [11]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [12]:
features = categorical + numerical

base_accuracy = train_and_score(features)
base_accuracy, round(base_accuracy, 2)

(0.645, 0.65)

## Question 5 — feature elimination

Difference = original accuracy − accuracy without the feature.

In [13]:
rows = []
for f in features:
    subset = [c for c in features if c != f]
    acc = train_and_score(subset)
    rows.append({'feature': f, 'accuracy': acc, 'diff': base_accuracy - acc})

df_elim = pd.DataFrame(rows).sort_values('diff')
df_elim

,feature,accuracy,diff
4,annual_income,0.724,-0.079
1,industry,0.645,0.000
3,location,0.645,0.000
2,employment_status,0.645,0.000
7,lead_score,0.644,0.001
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


In [14]:
df_elim[df_elim.feature.isin(['lead_source', 'number_of_courses_viewed', 'interaction_count'])]

,feature,accuracy,diff
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


## Question 6 — regularization strength `C`

In [15]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(features, C=C)
    print(f'C={C}: {round(acc, 3)}')

C=1e-06: 0.598
C=1e-05: 0.598
C=0.0001: 0.613
C=0.001: 0.645


## Answers

| # | Question | Answer |
|---|----------|--------|
| 1 | Mode for `industry` | `technology` |
| 2 | Biggest correlation | `interaction_count` and `lead_score` (0.916) |
| 3 | Biggest MI | `lead_source` (0.03) |
| 4 | Accuracy | 0.65 |
| 5 | Feature selection | `number_of_courses_viewed` (diff 0.002) |
| 6 | Parameter tuning | 0.001 (accuracy 0.645) |